In [1]:
import sys

if (path := "C:/Users/Tom/PycharmProjects/python-mechanics") not in sys.path:
    sys.path.append(path)

In [2]:
from mechanics import Quantity, print_styled_doc_string

Q_ = Quantity

# Equilibrium of a Body (Statics)
---

Notebook no. 1 explains how forces, moments and linear distributed loads are defined in package `python-mechanics`. In this notebook it is further explained and demonstrated:
- how to solve a system of external forces and moments acting on a beam.
- how to determine the internal loadings acting on a cross-section of the beam.

In [3]:
from mechanics import Position, Angle, Force, Moment, DistributedLoad1D, Beam

## Example 1

*Example 1.1 from Hibbeler, R. C. (2017). Mechanics of Materials in SI Units, 10th Edition*<br>

Determine the resultant internal loadings acting on the cross-section $C$ of a cantilevered beam.

<img src="./figures/example_1-1.png" style="max-width: 75%; height: auto; margin-left:auto; margin-right:auto">

<!-- ℹ️ Info Message -->
<div style="background-color:#e7f3fe; color:#084298; padding:10px; border-left:5px solid #2196F3; border-radius:4px;">
  ℹ️ <strong>Reference Frame</strong><br>
    By convention, we select the x-axis of our reference frame along the beam's center line, pointing to the right.
    The y-axis is oriented vertically upwards. As our reference frame is a right-handed Cartesian coordinate system,
    the selection of the x- and y-axes also determines the orientation of the z-axis. The z-axis is perpendicular to
    the screen, pointing out of the screen. A drawing of the reference frame conventions used throughout package
    <code>python-mechanics</code> can be found <a href="./right-handed-coordinate-system.pdf">here</a>.
</div>

### External Loadings Acting on the Beam

A linear distributed load $q$ acts on the beam as shown. At point $A$, which is chosen to be origin of our reference frame ($x$ = 0 m), the load $q$ has a magnitude of 300 N/m. At the other end of the beam (point $B$), which lies at $x$ = 3.6 m from the origin, the load has linearly decreased to zero. As the linear distributed load is acting vertically downwards, while the y-axis of our reference frame is pointing upwards, the magnitude of the load must be given a minus sign.

In [4]:
q = DistributedLoad1D(
    x_coords=Q_([0, 3.6], 'm'),
    loads=Q_([-300.0, 0.0], 'N / m'),
    name='q'
)

At point $A$, the beam is fixed into a wall. An unknown reaction moment and an unknown reaction force are exerted by the wall on the beam. We can see that the vector of the reaction moment about point A must be oriented along the Z-axis of our reference frame, and also that the reaction force must act in the direction of the Y-axis. So, only the magnitudes of the reaction moment and the reaction force are unknowns. 

> **Note**<br>
In a planar (2D) system of forces, three equations are available to solve for the static equilibrium of the system. This means, in general, that there cannot be more than three unknowns in the system to be able to solve this system. 

As the magnitudes of the reaction moment `M_A` and the reaction force `F_A` are not known, we give them a symbolic name.

In [5]:
M_A = Moment(
    magnitude='M_A',
    gamma=Angle(90),
    name='M_A'
)

In [6]:
F_A = Force(
    magnitude='F_A',
    theta=Angle(90),
    name='F_A'
)

The external forces and moments are acting on a beam, which is represented by the `Beam` class. In fact package `python-mechanics` has two implementations of a `Beam` class: a "basic version" and an "extended version". The "basic version" is implemented in module `statics.system.py`, while the "extended version" is implemented in module `strength.beam.py`. The "extended version" inherits from and builds upon the "basic version". When we import the `Beam` class from the main package `mechanics` as we did above, we are implicitly importing the "extended version".

In [7]:
print_styled_doc_string(Beam)

In [8]:
print_styled_doc_string(Beam.__init__)

At this stage, we only need the parameters `length` and `loadings` from the basic `Beam` class.

In [9]:
beam = Beam(
    length=Q_(3.6, 'm'),
    loadings=[q, M_A, F_A]
)

When instantiating the `Beam` class, the system is immediately solved for any unknown external forces or moments which are acting on the beam. The external forces, moments, and linear distributed loads are kept in dictionaries inside the object. The keys of the dictionaries are the names we've assigned to the forces, etc. on their instantiation. 

In [11]:
beam.ext_forces

{'F_A': <x: 0.0; y: 540.0; z: 0.0> N}

In [12]:
beam.ext_moments

{'M_A': <x: 0.0; y: 0.0; z: 648.0> N * m}

In [13]:
beam.ext_distrib_loads

{'q': <x: 0.0; y: -540.0; z: 0.0> N}

> **Note**<br>
The string representation of a `DistributedLoad1D` object is actually the string representation of its resultant force, which goes through the center of area of the distributed load. 

### Internal Loadings at the Specified Cross-Section

To find the resultant internal loadings on a cross-section of the beam, we use method `cut(...)`.

In [14]:
print_styled_doc_string(beam.cut)

In [15]:
F_int, M_int = beam.cut(x=Q_(1.2, 'm'))

On the (left-sided) cross-section at point $C$ in the figure above, the resultant internal force is:

In [16]:
print(F_int)

<x: 0.0; y: -240.0; z: 0.0> N


The minus sign indicates that the internal force is acting vertically downwards.

The resultant internal bending moment acting on the cross-section at point $C$ is:

In [17]:
print(M_int)

<x: 0.0; y: 0.0; z: -192.0> N * m


The minus sign indicates that the bending moment pulls on the top half of the (left-sided) cross-section and pushes on its bottom half.

Should we view the cross-section on the right side of the cut, we get:

In [18]:
F_int, M_int = beam.cut(x=Q_(1.2, 'm'), view='right')
print(F_int)
print(M_int)

<x: 0.0; y: 240.0; z: 0.0> N
<x: 0.0; y: 0.0; z: 192.0> N * m


The signs of the internal loadings on the right-sided cross-section are opposite to the signs of the internal loadings on the left-sided cross-section. On the right-sided cross-section the internal force is acting vertically upwards, but the internal bending moment is also pulling on the upper half of the right-sided cross-section, while pushing on its lower half. The internal loadings on the right-sided cross-section are exerted by the left side of the beam and vice versa. So, the sum of the internal loadings acting on both sides of a cross-section is zero, which makes sense as the beam is in static equilibrium. The internal vertical forces on both sides of the cross-section attempt to push the beam halves on either side of the cross-section vertically apart. The internal bending moments on both sides of the cross-section actually exert pressure on the top half of the cross-section, while simultaneously trying to pull the sides apart on the bottom half.

## Example 2

*Example 1.2 from Hibbeler, R. C. (2017). Mechanics of Materials in SI Units, 10th Edition*<br>

Determine the resultant internal loadings acting on the cross-section of the boom at point $E$.

<img src="./figures/example_1-2.png" style="max-width: 75%; height: auto; margin-left:auto; margin-right:auto">

The mass of the suspended engine is 500 kg.

### External Loadings Acting on the Beam

The hinge at point $A$ exerts a reaction force on the crane boom of which the magnitude and direction are unknown. Member $CD$ can be considered as a two-force member, i.e., it acts like a cable. This means that the reaction force exerted by member $CD$ on the crane boom has a known direction. So, we have a system of forces with three unknowns, which is solvable.  

**Force** $F_A$<br>
- magnitude unknown
- angle `theta` unknown
- angle `gamma` is zero (default), as the system is 2D and in the XY-plane of our reference frame
- position: origin (default)

In [19]:
F_A = Force(
    magnitude='F_A',
    theta='theta_A',
    position=Position(0, units='m')
)

**Force** $F_C$<br>
- magnitude unknown
- angle `theta` not specified, but can be determined with `Angle.create(...)`
- angle `gamma` is zero
- position: $x$ = 2 m 

In [20]:
F_C = Force(
    magnitude='F_C',
    theta=Angle.create(v=1.5, h=2, quadrant=2),
    position=Position(2, units='m')
)

**Force** $F_B$<br>
- magnitude: mass engine = 500 kg
- angle `theta` is -90°, as the weight of the engine is acting vertically downwards
- angle `gamma` is zero
- position: $x$ = 3 m

In [21]:
F_B = Force(
    magnitude=9.81 * 500,
    theta=Angle(-90),
    position=Position(3, units='m'),
    name='F_B'
)

Now create the `Beam` object with the applied external loadings. Unknown forces/moments will determined while the `Beam` object is created.

In [22]:
crane_boom = Beam(
    length=Q_(3, 'm'),
    loadings=[F_A, F_B, F_C]
)

Show external loadings applied to the beam:

In [24]:
crane_boom.ext_forces

{'F_A': <x: 9810.0; y: -2452.5; z: 0.0> N,
 'F_B': <x: 0.0; y: -4905.0; z: 0.0> N,
 'F_C': <x: -9810.0; y: 7357.5; z: 0.0> N}

In [25]:
crane_boom.ext_moments

{}

There are no external moments acting on the beam. 

### Internal Loadings at the Specified Cross-Section

To determine the internal loadings at point $E$ ($x$ = 1 m), the crane boom is cut at point $E$. By default (if we don't specify parameter `view`), we will look at the left-sided cross-section. 

In [26]:
F_int, M_int = crane_boom.cut(x=Q_(1, 'm'))
print(F_int)
print(M_int)

<x: -9810.0; y: 2452.5; z: 0.0> N
<x: 0.0; y: 0.0; z: -2452.5> N * m


As indicated by the minus sign, an axial force (along the longitudinal x-axis of the crane boom and in the negative x-direction) pushes on the left-sided cross-section (compressive force), while a vertically upward internal force acts in the plane of the cross-section (shear force). Also, an internal bending moment pulls on the top half and pushes on the bottom half of the cross-section. 

To get the magnitude of the resultant internal force in units of kN and the magnitude of the internal bending moment in units of kN$\cdot$m:

In [27]:
print(F_int.magnitude.to('kN'))
print(M_int.magnitude.to('kN * m'))

10.111916546827313 kilonewton
2.4525 kilonewton * meter


## Example 3

*Example 1.3 from Hibbeler, R. C. (2017). Mechanics of Materials in SI Units, 10th Edition*<br>

Determine the resultant internal loadings acting on the cross-section at $G$ of the beam. Each joint is pin connected. 

<img src="./figures/example_1-3.png" style="max-width: 75%; height: auto; margin-left:auto; margin-right:auto">

When we look at the beam, which is supported by a truss structure, we see that four unknowns are present:
- only the magnitude of reaction force $R_A$
- only the magnitude of reaction force $R_D$
- both the magnitude and direction of reaction force $R_E$

However, if we look at the complete system, we see that there are only three unknowns, so this system is solvable. The unknowns are:
- the magnitude of reaction force $R_C$
- both the magnitude and direction of reaction force $R_E$ 

If we first solve the system, then we will know the reaction force at point $E$ of our beam. To do this, we can use the class `System`.

In [28]:
from mechanics import System

In [29]:
print_styled_doc_string(System)

In [30]:
print_styled_doc_string(System.__init__)

As with the `Beam` class (which is actually derived from class `System`), we need to pass a list with the forces and/or moments that act on the system. 

In [31]:
F_A = Force(
    magnitude=1500,
    theta=Angle(-90),
    position=Position(0, 0, units='m'),
    units='N',
    name='F_A'
)

q = DistributedLoad1D(
    x_coords=Q_([2, 5], 'm'),
    loads=Q_([-600, 0], 'N / m'),
    name='q'
)

R_C = Force(
    magnitude='R_C',
    theta=Angle(0),
    position=Position(5, 1.5, units='m')
)

R_E = Force(
    magnitude='R_E',
    theta='theta_E',
    position=Position(5.0, 0.0, units='m')
)

Once the known and unknown forces have been defined, we can create a `System` object and then solve the system for the unknown forces.

In [32]:
sys = System([F_A, R_C, R_E, q])
solutions = sys.solve()

In [33]:
print_styled_doc_string(sys.solve)

In [34]:
for name, solution in solutions.items():
    print(f"{name}: {solution}")

R_C: <x: 6200.0; y: 0.0; z: 0.0> N
R_E: <x: -6200.0; y: 2400.0; z: 0.0> N


From here, we can continue in the same way as in the previous examples and use the `Beam` class. We define the unknown reaction forces in point $A$ and in point $D$, and we will use the solution for the reaction force in point $E$.

In [35]:
R_A = Force(
    magnitude='R_A',
    theta=Angle.create(3, 4, quadrant=1),
    position=Position(0, units='m')
)

R_D = Force(
    magnitude='R_D',
    position=Position(2, units='m'),
    theta=Angle(90)
)

F_E = solutions['R_E']

Now, we create the `Beam` object, and then we cut the beam at point $G$.

In [36]:
beam = Beam(
    length=Q_(5, 'm'),
    loadings=[F_A, q, F_E, R_A, R_D]
)

iF_G, iM_G = beam.cut(x=Q_(1, 'm'))

print(iF_G)
print(iM_G)

<x: -6200.0; y: -3150.0; z: 0.0> N
<x: 0.0; y: 0.0; z: 3150.0> N * m
